In [13]:

!pip  install seaborn matplotlib pandas numpy
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
# Import du jeu de donnée asssemblé mais non nettoyé
df = pd.read_csv("../data/raw/accidents_2019_2023.csv")


C:\Users\nvann\AppData\Local\Temp\ipykernel_34924\1026929783.py:7: DtypeWarning: Columns (42,45,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/accidents_2019_2023.csv")


In [14]:
import numpy as np
import pandas as pd

# Étape 1 — Nettoyer les données (remplacer -1 par NaN)
df[['secu1', 'secu2']] = df[['secu1', 'secu2']].replace(-1, np.nan)

# Étape 2 — Fonction pour calculer la distribution conditionnelle par colonne
def get_proba_by_col(df, colname, group_cols=['catu', 'catv']):
    temp = df[df[colname].notna()].copy()
    temp['profil'] = temp[group_cols].astype(str).agg('_'.join, axis=1)
    return (
        temp.groupby('profil')[colname]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
    )

# Étape 3 — Calcul des distributions conditionnelles séparées
proba_secu1 = get_proba_by_col(df, 'secu1')
proba_secu2 = get_proba_by_col(df, 'secu2')


# Étape 4 — Fonction d’imputation basée sur les distributions
def imputer_secu(df, col, proba_df, profil_cols):
    def impute(row):
        if pd.notna(row[col]):
            return row[col]
        profil_key = "_".join([str(row[c]) for c in profil_cols])
        if profil_key not in proba_df.index:
            return np.nan  # fallback possible ici
        p = proba_df.loc[profil_key]
        return np.random.choice(p.index, p=p.values)
    return df.apply(impute, axis=1)

# Étape 5 — Imputation indépendante des 3 colonnes
df['secu1_corr'] = imputer_secu(df, 'secu1', proba_secu1, ['catu', 'catv'])
df['secu2_corr'] = imputer_secu(df, 'secu2', proba_secu2, ['catu', 'catv'])


# Étape 6 — Regrouper les équipements corrigés dans une seule liste
def regrouper_equipements(row):
    return [x for x in [row['secu1_corr'], row['secu2_corr']] if pd.notna(x)]

df['equipements'] = df.apply(regrouper_equipements, axis=1)


In [15]:
#encodage de la liste 'equipement'regrouper_equipements
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
equip_ohe = pd.DataFrame(mlb.fit_transform(df['equipements']), 
                         columns=[f'eq_{int(c)}' for c in mlb.classes_],
                         index=df.index)

# Fusion avec ton DataFrame principal
df = pd.concat([df, equip_ohe], axis=1)

In [ ]:
création des colonnes de sécurité d'un csv pour validation
#df[['catu', 'catv', 'secu1', 'secu2', 'secu3', 'secu1_corr', 'secu2_corr', 'secu3_corr', 'equipements', 'eq_1','eq_2','eq_3','eq_4','eq_5','eq_6','eq_7','eq_8','eq_9']].to_csv("../data/processed/accidents_2019_2023_secu.csv", index=False)

In [16]:
#Suppression des colonnes inutiles
df.drop(columns=['secu1', 'secu2','secu3','secu1_corr','secu1_corr','secu1_corr', 'equipements'], inplace=True)

In [ ]:
#df.head()


,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,...,eq_0,eq_1,eq_2,eq_3,eq_4,eq_5,eq_6,eq_7,eq_8,eq_9
0,201900000001,138 306 524,B01,2,7,0,2,5,23,1,...,1,1,0,0,0,0,0,0,0,0
1,201900000001,138 306 524,B01,2,7,0,2,5,23,1,...,1,1,0,0,0,0,0,0,0,0
2,201900000001,138 306 525,A01,2,17,1,0,3,11,1,...,1,1,0,0,0,0,0,0,0,0
3,201900000002,138 306 523,A01,1,7,4,0,1,0,1,...,1,1,0,0,0,0,0,0,0,0
4,201900000003,138 306 520,A01,1,7,0,2,1,2,1,...,1,1,0,0,0,0,0,0,0,0
